In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.customers AS
SELECT customer_id, customer_unique_id,
customer_zip_code_prefix AS zip_prefix,
lower(trim(customer_city)) AS city,
upper(trim(customer_state)) AS state,
_ingested_at
FROM (
    SELECT *, row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at) AS rn
    FROM workspace.olist_bronze.bronze_customers
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.sellers AS
SELECT seller_id,
seller_zip_code_prefix AS zip_prefix,
lower(trim(seller_city)) AS city,
upper(trim(seller_state)) AS state,
_ingested_at
FROM(
    SELECT *, ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY _ingested_at) AS rn
    FROM workspace.olist_bronze.bronze_sellers
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.category_translation AS 
SELECT DISTINCT 
lower(trim(product_category_name)) AS category_name_pt,
lower(trim(product_category_name_english)) AS category_name_en
FROM workspace.olist_bronze.bronze_product_category_name_translation;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.products AS
SELECT product_id,
NULLIF(lower(trim(product_category_name)),'') AS category_name_pt,
try_cast(product_photos_qty AS INT) AS photos_qty,
try_cast(product_weight_g AS INT) AS weight_g,
try_cast(product_length_cm AS INT) AS length_cm,
try_cast(product_height_cm AS INT) AS height_cm,
try_cast(product_width_cm AS INT) AS width_cm,
_ingested_at
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY _ingested_at) AS rn
    FROM workspace.olist_bronze.bronze_products
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.orders AS 
SELECT order_id, customer_id,
lower(trim(order_status)) AS order_status,
try_cast(order_purchase_timestamp AS TIMESTAMP) AS purchased_at,
try_cast(order_approved_at AS TIMESTAMP) AS approved_at,
try_cast(order_delivered_carrier_date AS TIMESTAMP) AS shipped_at,
try_cast(order_delivered_customer_date AS TIMESTAMP) AS delivered_at,
try_cast(order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_at,
_ingested_at
FROM(
    SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _ingested_at) AS rn
    FROM workspace.olist_bronze.bronze_orders
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.order_items AS
SELECT order_id,
       try_cast(order_item_id AS INT) AS order_item_id,
       product_id, seller_id,
       try_cast(shipping_limit_date AS TIMESTAMP) AS shipping_limit_at,
       try_cast(price AS DECIMAL(10,2)) AS price,
       try_cast(freight_value AS DECIMAL(10,2)) AS freight_value,
       _ingested_at
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id, order_item_id ORDER BY _ingested_at) AS rn
  FROM workspace.olist_bronze.bronze_order_items
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.payments AS
SELECT order_id,
try_cast(payment_sequential AS INT) AS payment_sequential,
lower(trim(payment_type)) AS payment_type,
try_cast(payment_installments AS INT) AS installments,
try_cast(payment_value AS DECIMAL(10,2)) AS payment_value,
_ingested_at
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id, payment_sequential ORDER BY _ingested_at) AS rn
    FROM workspace.olist_bronze.bronze_order_payments
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_silver_sql.reviews AS 
SELECT review_id, order_id,
try_cast(review_score AS INT) AS review_score,
try_cast(review_creation_date AS TIMESTAMP) AS created_at,
try_cast(review_answer_timestamp AS TIMESTAMP) AS answered_at,
_ingested_at
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY review_id, order_id
        ORDER BY try_cast(review_answer_timestamp AS TIMESTAMP) DESC) AS rn
        FROM workspace.olist_bronze.bronze_order_reviews
    )
    WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'customers' AS tbl, COUNT(*) AS rows FROM workspace.olist_silver_sql.customers
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.olist_silver_sql.sellers
UNION ALL SELECT 'products', COUNT(*) FROM workspace.olist_silver_sql.products
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.olist_silver_sql.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.olist_silver_sql.order_items
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.olist_silver_sql.payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.olist_silver_sql.reviews
UNION ALL SELECT 'category_translation', COUNT(*) FROM workspace.olist_silver_sql.category_translation;

tbl,rows
customers,99441
sellers,3095
products,32951
orders,99441
order_items,112650
payments,103886
reviews,99224
category_translation,71


In [0]:
%sql
SELECT COUNT(*) AS orders,
COUNT(purchased_at) AS has_purchase_date,
COUNT(delivered_at) AS has_delivery_date
FROM workspace.olist_silver_sql.orders;

orders,has_purchase_date,has_delivery_date
99441,99441,96476


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
olist_bronze
olist_gold_py
olist_gold_sql
olist_silver_py
olist_silver_sql
property


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_silver_sql;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_silver_py;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_gold_sql;